# Matching 5D-ntuple cells to Calo-ntuple cells

We have two ntuple types:

- **5D ntuples** (SLAC, tree `ntuple`): store a per-event *subset* of calorimeter cells.
  They carry the offline `Cell_ID` but **no hash ID**, so cells cannot be identified directly.
- **Calo ntuples** (ours, tree `analysis`): store **all** calorimeter cells every event,
  including `cell_hashID`.

Both store the cell-centre position `(x, y, z)`, `(eta, phi)` and the `sampling` layer,
all derived from the same detector description. **Algorithm**: build one KD-tree per
sampling layer from the Calo ntuple's full cell list, then match each 5D cell to the
nearest calo cell *in the same sampling* by `(x, y, z)`. Because the coordinates come
from the same geometry, matched distances are exactly zero (float-identical), making the
match unambiguous. From the matches we accumulate a persistent `Cell_ID -> hashID`
lookup table, so the geometric match only ever has to run once.

The algorithm lives in [`cell_matching.py`](cell_matching.py) — import it with
`from cell_matching import CaloCellGeometry, CellMatcher, CellIDMap, match_5d_event`.

In [1]:
import glob, time
import numpy as np
import awkward as ak
import uproot
%matplotlib inline
import matplotlib.pyplot as plt

import sys, os
sys.path.insert(0, os.path.abspath(".."))  # cell_matching.py lives at the repo root
from cell_matching import (CaloCellGeometry, CellMatcher, CellIDMap,
                           match_5d_event, sampling_name)

In [2]:
FIVED_FILES = sorted(glob.glob("/storage/mxg1065/ttbar_100GB/*.root"))
CALO_FILE = "/storage/mxg1065/input_data/ttbar_100events.root"

t5 = uproot.open(FIVED_FILES[0])["ntuple"]
tc = uproot.open(CALO_FILE)["analysis"]
print(f"5D:   {len(FIVED_FILES)} files; using {FIVED_FILES[0].split('/')[-1]}: {t5.num_entries} events")
print(f"Calo: {CALO_FILE.split('/')[-1]}: {tc.num_entries} events")

5D:   17 files; using user.bbullard.50733453._000011.ntuple.root: 20000 events
Calo: ttbar_100events.root: 100 events


## 1. The Calo ntuple's cell list is the full, static detector

Every event stores the same 187,652 cells with `cell_hashID = 0..187651` in order —
so a single event gives us the complete calorimeter geometry, and any per-event calo
array can be indexed directly by hashID.

In [3]:
assert CaloCellGeometry.is_static_across_events(tc, entries=(0, 1, 2)), \
    "calo cell list differs between events - would need per-event geometry"

geo = CaloCellGeometry.from_tree(tc, entry=0)
print(f"full calo cell list: {geo.n} cells, "
      f"{len(np.unique(geo.sampling))} sampling layers")
print(f"hashID is dense 0..N-1 in order: {np.array_equal(geo.hash_id, np.arange(geo.n))}")

full calo cell list: 187652 cells, 24 sampling layers
hashID is dense 0..N-1 in order: True


## 2. Build the matcher

One KD-tree per sampling layer over cell-centre `(x, y, z)`. A query cell can only
match a calo cell in its own layer, so layers with cells at the same `(eta, phi)`
(e.g. the Tile layers) can never confuse the match. The 1 mm tolerance is far below
the smallest cell pitch (~4 mm EMB1 strips) and far above float32 coordinate rounding
(< 0.001 mm).

In [4]:
t0 = time.perf_counter()
matcher = CellMatcher(geo, tolerance_mm=1.0)
print(f"built KD-trees for samplings {matcher.samplings}")
print(f"total {geo.n} cells indexed in {time.perf_counter() - t0:.2f} s")

built KD-trees for samplings [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
total 187652 cells indexed in 0.06 s


## 3. Match a few 5D events

(A few events is enough to demonstrate — and matching is ~ms per event anyway;
the one-off cost is reading the calo geometry above.)

In [5]:
N_DEMO = 5
id_map = CellIDMap()
results = []
all_dist = []
for entry in range(N_DEMO):
    t0 = time.perf_counter()
    cell_ids, res = match_5d_event(t5, matcher, entry)
    dt = time.perf_counter() - t0
    n_new = id_map.update(cell_ids, res)
    results.append((entry, cell_ids, res))
    all_dist.append(res.distance[res.matched])
    dmax = res.distance[res.matched].max() if res.n_matched else float("nan")
    print(f"event {entry}: {res.n_matched:4d}/{res.n:4d} matched ({100 * res.match_rate:6.2f}%), "
          f"max dist {dmax:.3g} mm, "
          f"{len(res.duplicate_calo_indices())} duplicate targets, "
          f"{n_new:4d} new Cell_IDs mapped  [{dt * 1e3:.0f} ms]")

all_dist = np.concatenate(all_dist)
print(f"\nmatch distance over all {len(all_dist)} matched cells: "
      f"min={all_dist.min():.3g}  median={np.median(all_dist):.3g}  max={all_dist.max():.3g} mm")
print(f"Cell_ID -> hashID map: {id_map.n} entries, {len(id_map.conflicts)} conflicts")

event 0:  810/ 810 matched (100.00%), max dist 0 mm, 0 duplicate targets,  810 new Cell_IDs mapped  [9 ms]
event 1:  422/ 422 matched (100.00%), max dist 0 mm, 0 duplicate targets,  417 new Cell_IDs mapped  [8 ms]
event 2:  621/ 621 matched (100.00%), max dist 0 mm, 0 duplicate targets,  616 new Cell_IDs mapped  [8 ms]
event 3:  636/ 636 matched (100.00%), max dist 0 mm, 0 duplicate targets,  612 new Cell_IDs mapped  [8 ms]
event 4:  603/ 603 matched (100.00%), max dist 0 mm, 0 duplicate targets,  590 new Cell_IDs mapped  [8 ms]

match distance over all 3092 matched cells: min=0  median=0  max=0 mm
Cell_ID -> hashID map: 3045 entries, 0 conflicts


In [6]:
entry, cell_ids, res = results[-1]
print(f"per-sampling summary, event {entry}:\n")
print(res.summary())

per-sampling summary, event 4:

matched 603/603 cells (100.00%)  [metric=xyz, tolerance=1mm]
sampling       n_query n_match    median_d       max_d
PreSamplerB         42      42           0           0
EMB1                 9       9           0           0
EMB2                74      74           0           0
EMB3                 4       4           0           0
PreSamplerE         12      12           0           0
EME1               157     157           0           0
EME2               180     180           0           0
EME3                17      17           0           0
HEC0                14      14           0           0
HEC1                 5       5           0           0
HEC2                 4       4           0           0
HEC3                 1       1           0           0
TileBar0            32      32           0           0
TileBar1             9       9           0           0
TileGap2             2       2           0           0
TileGap3            22     

In [7]:
# a few example matches from event 0
entry, cell_ids, res = results[0]
arr = t5.arrays(["Cell_eta", "Cell_phi"], entry_start=entry, entry_stop=entry + 1)
eta5 = ak.to_numpy(arr["Cell_eta"][0])
phi5 = ak.to_numpy(arr["Cell_phi"][0])
print(f"{'Cell_ID':>12}  {'sampling':<12}{'eta':>8}{'phi':>8}      {'hashID':>8}{'dist [mm]':>11}")
for i in np.where(res.matched)[0][:10]:
    print(f"{cell_ids[i]:>12}  {sampling_name(res.sampling[i]):<12}"
          f"{eta5[i]:>8.3f}{phi5[i]:>8.3f}  ->  {res.hash_id[i]:>8}{res.distance[i]:>11.3g}")

     Cell_ID  sampling         eta     phi        hashID  dist [mm]
   740294754  EME2          -2.562  -1.418  ->        49          0
   740296756  EME2          -2.963   2.603  ->       282          0
   740296804  EME2          -2.962  -1.319  ->       306          0
   740297782  EME2          -3.164   2.701  ->       411          0
   740297784  EME2          -3.164   2.800  ->       412          0
   742393458  EME3          -2.860  -0.632  ->       697          0
   742393460  EME3          -2.859  -0.534  ->       698          0
   746587240  PreSamplerE   -1.574  -1.125  ->      1076          0
   746587696  PreSamplerE   -1.599   2.409  ->      1112          0
   746589272  PreSamplerE   -1.674  -1.910  ->      1324          0


## 4. Validation

Three internal-consistency checks that don't depend on the two files containing the
same events:

1. matched pairs must also agree in `(eta, phi)` (an independent coordinate pair),
2. within an event no two 5D cells may claim the same calo cell (checked above:
   "duplicate targets"),
3. across events the accumulated `Cell_ID -> hashID` map must be consistent
   (0 conflicts above) and injective.

In [8]:
max_deta = max_dphi = 0.0
for entry, cell_ids, res in results:
    arr = t5.arrays(["Cell_eta", "Cell_phi"], entry_start=entry, entry_stop=entry + 1)
    eta5 = ak.to_numpy(arr["Cell_eta"][0])
    phi5 = ak.to_numpy(arr["Cell_phi"][0])
    j = res.index[res.matched]
    deta = np.abs(geo.eta[j] - eta5[res.matched])
    dphi = np.abs(np.angle(np.exp(1j * (geo.phi[j] - phi5[res.matched]))))
    max_deta = max(max_deta, deta.max())
    max_dphi = max(max_dphi, dphi.max())
print(f"matched pairs also agree in eta/phi: max|d_eta| = {max_deta:.3g}, max|d_phi| = {max_dphi:.3g}")
print(f"map is injective (distinct Cell_IDs -> distinct hashIDs): {id_map.is_injective()}")
print(f"map is clean (no ID conflicts, no duplicate targets): {id_map.clean}")

matched pairs also agree in eta/phi: max|d_eta| = 0, max|d_phi| = 0
map is injective (distinct Cell_IDs -> distinct hashIDs): True
map is clean (no ID conflicts, no duplicate targets): True


### Energy cross-check (only possible if the files share events)

If the two files were produced from the same events, matched cells must have identical
energies (5D stores GeV, calo stores MeV). Check the event-number overlap:

In [9]:
ev5 = np.asarray(t5["eventNumber"].array(library="np")).ravel()
evc = np.asarray(tc["EventNumber"].array(library="np")).ravel()
common = np.intersect1d(ev5, evc)
print(f"5D file eventNumber range [{ev5.min()}, {ev5.max()}], "
      f"calo file [{evc.min()}, {evc.max()}]; common events: {len(common)}")

if len(common):
    n = int(common[0])
    i5 = int(np.where(ev5 == n)[0][0])
    ic = int(np.where(evc == n)[0][0])
    cell_ids_n, res_n = match_5d_event(t5, matcher, i5)
    e5 = ak.to_numpy(t5["Cell_e"].array(entry_start=i5, entry_stop=i5 + 1)[0])
    ec = ak.to_numpy(tc["cell_e"].array(entry_start=ic, entry_stop=ic + 1)[0])
    m = res_n.matched
    e5m = e5[m] * 1000.0  # GeV -> MeV
    ecm = ec[res_n.index[m]]
    ratio = ecm / np.where(e5m == 0, np.nan, e5m)
    print(f"event {n}: median calo/5D cell-energy ratio = {np.nanmedian(ratio):.4f} (expect 1.0)")
    plt.figure(figsize=(5, 5))
    plt.scatter(e5m, ecm, s=4)
    lims = [min(e5m.min(), ecm.min()), max(e5m.max(), ecm.max())]
    plt.plot(lims, lims, "r--", lw=1)
    plt.xlabel("5D Cell_e x 1000 [MeV]")
    plt.ylabel("calo cell_e [MeV]")
    plt.title(f"matched-cell energies, event {n}")
else:
    print("-> the two files do not contain the same events, so an energy cross-check is\n"
          "   not possible here. It is also not needed: zero match distances plus the\n"
          "   independent eta/phi agreement already identify the cells unambiguously.")

5D file eventNumber range [5000001, 5021000], calo file [0, 0]; common events: 0
-> the two files do not contain the same events, so an energy cross-check is
   not possible here. It is also not needed: zero match distances plus the
   independent eta/phi agreement already identify the cells unambiguously.


## 5. The persistent `Cell_ID -> hashID` map

Matched events accumulate into a lookup table. Save it once, then translate any 5D
event's `Cell_ID`s to hashIDs in microseconds with no geometry at all. IDs not yet in
the table return -1 — run the geometric matcher on those events to extend the table
(over enough events it converges to every cell that ever fires).

In [10]:
id_map.save("cellid_to_hashid.npz")
reloaded = CellIDMap.load("cellid_to_hashid.npz")
print(f"saved + reloaded map with {reloaded.n} entries")

# an event the map has never seen
entry = N_DEMO
t0 = time.perf_counter()
cell_ids_new, res_new = match_5d_event(t5, matcher, entry)
dt_geo = time.perf_counter() - t0

t0 = time.perf_counter()
hids_lut = reloaded.lookup(cell_ids_new)
dt_lut = time.perf_counter() - t0

known = hids_lut >= 0
both = known & res_new.matched
agree = np.array_equal(hids_lut[both], res_new.hash_id[both])
print(f"event {entry}: {known.sum()}/{len(hids_lut)} Cell_IDs already in the map "
      f"(from just {N_DEMO} events); lookup agrees with geometric match: {agree}")
print(f"timing: geometric match {dt_geo * 1e3:.1f} ms vs table lookup {dt_lut * 1e6:.0f} us")

saved + reloaded map with 3045 entries
event 5: 13/516 Cell_IDs already in the map (from just 5 events); lookup agrees with geometric match: True
timing: geometric match 8.5 ms vs table lookup 344 us


## 6. Fallback: matching without `(x, y, z)`

If a dataset only has `(eta, phi, sampling)`, `match_eta_phi` uses the metric
`sqrt(d_eta^2 + (2 sin(d_phi/2))^2)` (phi-wrap safe) per sampling layer:

In [11]:
arr = t5.arrays(["Cell_eta", "Cell_phi", "Cell_sampling"], entry_start=0, entry_stop=1)
res_ep = matcher.match_eta_phi(arr["Cell_eta"][0], arr["Cell_phi"][0], arr["Cell_sampling"][0])
res_xyz = results[0][2]
both = res_ep.matched & res_xyz.matched
agree = (res_ep.hash_id[both] == res_xyz.hash_id[both]).mean() if both.any() else float("nan")
print(f"eta-phi fallback: {res_ep.n_matched}/{res_ep.n} matched; "
      f"agreement with xyz matching: {100 * agree:.2f}%")

eta-phi fallback: 810/810 matched; agreement with xyz matching: 100.00%


## Summary

```python
from cell_matching import CaloCellGeometry, CellMatcher, CellIDMap, match_5d_event

geo     = CaloCellGeometry.from_tree(uproot.open(CALO_FILE)["analysis"])
matcher = CellMatcher(geo)                       # KD-tree per sampling layer
id_map  = CellIDMap.load("cellid_to_hashid.npz") # or build via id_map.update(...)

cell_ids, res = match_5d_event(t5, matcher, entry)  # geometric match, ~ms
hash_ids      = id_map.lookup(cell_ids)             # table lookup, ~us, -1 = unknown
```

- Matching is **exact**: cell centres are float-identical between the two ntuple
  types, so every matched distance is 0 and the 1 mm tolerance leaves no ambiguity.
- `res.index` also indexes directly into any per-event calo array
  (`cell_e`, `cell_truth`, ...), since the calo cell list is static and
  hash-ordered — that's how you pull calo-side quantities for 5D cells.
- Extend `cellid_to_hashid.npz` by running more events through
  `id_map.update(cell_ids, res)`. Inconsistencies are recorded as the map fills:
  a Cell_ID re-matched to a different hashID lands in `map.conflicts` (and is
  dropped, so `lookup` returns -1 for it); two Cell_IDs claiming the same hashID
  land in `map.duplicate_targets`. Both survive `save`/`load`, and `map.clean`
  should stay `True`.